### Importing Files

In [43]:
import torch 
import torch_geometric
import numpy as np
import pandas as pd
import scipy
import mne

from torch_geometric.nn import MessagePassing
import torch.nn.functional as F
from torch_geometric.utils import dense_to_sparse, softmax

### Common declarations

In [ ]:
criterion = torch.nn.CrossEntropyLoss()

## GCN (Trial)

### Basic Execution

In [45]:
X_sample = torch.randn(5,6) # Taking random values for each node's features
labels = torch.tensor([0, 0, 0, 1, 1])

coords = torch.tensor([
    [0.0,  0.7,  0.0],   # Fz
    [0.3,  0.5,  0.0],   # F3
   [-0.3,  0.5,  0.0],   # F4
    [0.0,  0.0,  0.9],   # Cz
    [0.5,  0.0,  0.5],   # C4
])

# Coordinates of that help us choose whether nodes should be considered neighbors or not. 

node_names = ["Fz", "F3", "F4", "CZ", "C4"]

W = torch.randn(6,4) 

A = torch.zeros(5,5)

def check_neighbors():
    for electrode in range(len(coords)): # Running over each electrode's coordinates
        for neighbor in range(len(coords)):
            if neighbor != electrode:
                distance_from_neighbor = 0.0
                for coord in range(len(coords[electrode])): # Running over each coordinate of electrode and the comparison electrode
                    distance_from_neighbor += (coords[electrode][coord] - coords[neighbor][coord]) ** 2
                
                distance_from_neighbor = distance_from_neighbor ** 0.5

            else:
                continue

            # print(f"Distance between: {node_names[electrode]} and {node_names[neighbor]} is: {distance_from_neighbor}")

            if distance_from_neighbor < 0.61:
                print(f"{node_names[electrode]} and {node_names[neighbor]} are neighbours")
            
            # else:
            #     print(f"{node_names[electrode]} and {node_names[neighbor]} are not neighbours")
            #     print(distance_from_neighbor)

### GCN Layer

In [46]:
class GCNLayer(torch.nn.Module):
    def __init__(self, in_features, out_features):
        super().__init__()
        self.W = torch.nn.Parameter(torch.randn(in_features, out_features))
        self.in_features = in_features
        self.out_features = out_features
    
    def forward(self, X, A):
        degrees = A.sum(dim=1)
        D_inv = torch.diag(1.0/degrees)

        normalized_A = torch.matmul(D_inv, A)
        normalized_result = torch.matmul(normalized_A, X)

        transformed_results = torch.matmul(normalized_result, self.W)

        return transformed_results

### Working of GCN layer

In [47]:
#Creating adjacency matrix
diff = coords.unsqueeze(0) - coords.unsqueeze(1) # Converts tensors to single dimensional tensors
dist = (diff ** 2).sum(dim=-1) ** 0.5  
A = (dist < 0.61).float() # Condition for each value in the tensor. if true => 1., if not => 0.
A.fill_diagonal_(0) # Fills diagonals with zeros

# Fixing degree-scale problem
identity_matrix = torch.eye(5)
A_hat = A + identity_matrix

edge_index, _ = dense_to_sparse(A_hat)

In [48]:
gcn = GCNLayer(6, 4)
optimizer_gcn = torch.optim.Adam(gcn.parameters(), lr = 0.01)

# for epoch in range(100):
#     optimizer.zero_grad()
#     out = gcn(X_sample, A_hat)
#     loss = criterion(out, labels)
#     loss.backward()
#     optimizer.step()

    # if epoch % 10 == 0:
    #     print(f"Epoch {epoch}, Loss: {loss.item()}")

## GAT

### GAT Layer

In [49]:
class GATLayer(MessagePassing):
    def __init__(self, in_features, out_features):
        super().__init__(aggr='add')
        self.W = torch.nn.Linear(in_features, out_features, bias = False) #Wraps X @ W into cleaner syntax
        self.a = torch.nn.Parameter(torch.randn(2 * out_features)) # fEATURE vector for the two nodes

    def forward(self, X, edge_index):
        X = self.W(X)
        return self.propagate(edge_index, X=X, size=(X.shape[0], X.shape[0]))
       
    def message(self, X_i, X_j, index):
        cont_msg = torch.cat([X_i, X_j], dim=1)
        att_score = (self.a * cont_msg).sum(dim=-1, keepdim=True)
        final_score = softmax(att_score, index)

        return X_j * final_score

    def update(self, aggr_out):
        return aggr_out

### Working

In [50]:
# gat= GATLayer(6,4)
# optimizer = torch.optim.Adam(gat.parameters(), lr = 0.01)

# for epoch in range(1000):
#     optimizer.zero_grad()
#     out = gat(X_sample, edge_index)
#     loss = criterion(out, labels)
#     loss.backward()
#     optimizer.step()

#     if epoch % 200 == 0:
#         print(f"Epoch {epoch}, Loss: {loss.item()}")

# BrainSync Model

### Single Layer

In [51]:
class BrainSync(torch.nn.Module):
    def __init__(self, in_features, out_features):
        super().__init__()
        self.gat = GATLayer(in_features, out_features)
        self.lin_layer = torch.nn.Linear(out_features, 2)

    def forward(self, X, edge_index):
        out = self.gat(X, edge_index)
        mean = torch.mean(out, dim=0)
        classes = self.lin_layer(mean)

        return classes

In [52]:
# X = torch.randn(64, 6)
# edge_index_final = torch.randint(0, 64, (2, 200))
# labels_final = torch.randint(0, 2, (1,)).squeeze() #for the graph not the features

# sync = BrainSync(6, 4)

# optimizer = torch.optim.Adam(sync.parameters(), lr = 0.01)

# for epoch in range(100):
#     optimizer.zero_grad()
#     out = sync(X, edge_index_final)
#     loss = criterion(out, labels_final)
#     loss.backward()
#     optimizer.step()

#     # if epoch % 10 == 0:
#     #     print(f"Epoch {epoch}, Loss: {loss.item()}")

### 2-Layered

In [53]:
class BrainSync_2(torch.nn.Module):
    def __init__(self, in_features, out_features):
        super().__init__()
        self.gat_layer1 = GATLayer(in_features, 4)
        self.gat_layer2 = GATLayer(4, out_features)
        self.lin_layer = torch.nn.Linear(out_features, 2)

    def forward(self, X, edge_index):
        res1 = self.gat_layer1(X, edge_index)
        res2 = self.gat_layer2(res1, edge_index)
        mean = torch.mean(res2, dim=0)
        classes = self.lin_layer(mean)
        
        return classes

In [54]:
# sync_2layered = BrainSync_2(6,4)
# optimizer = torch.optim.Adam(sync_2layered.parameters(), lr = 0.01)
# X = torch.randn(64, 6)
# edge_index_final = torch.randint(0, 64, (2, 200))
# labels_final = torch.randint(0, 2, (1,)).squeeze() #for the graph not the features

# for epoch in range(100):
#     optimizer.zero_grad()
#     out = sync_2layered(X, edge_index_final)
#     loss = criterion(out, labels_final)
#     loss.backward()
#     optimizer.step()

#     # if epoch % 10 == 0:
#     #     print(f"Epoch {epoch}, Loss: {loss.item()}")

### 8-Layered

In [55]:
class BrainSync_8(torch.nn.Module):
    def __init__(self, in_features, out_features):
        super().__init__()
        
        self.layer1 = GATLayer(in_features, 4)
        self.layer2 = GATLayer(4, 4)
        self.layer3 = GATLayer(4, 4)
        self.layer4 = GATLayer(4, 4)
        self.layer5 = GATLayer(4, 4)
        self.layer6 = GATLayer(4, 4)
        self.layer7 = GATLayer(4, 4)
        self.layer8 = GATLayer(4, out_features)

        self.proj = torch.nn.Linear(in_features, 4)
        self.lin_layer = torch.nn.Linear(out_features, 2)


    def forward(self, X, edge_index):
        res1 = self.layer1(X, edge_index)
        prev_proj = self.proj(X)
        res1 = res1 + prev_proj
        res2 = self.layer2(res1, edge_index)
        res2 = res2 + res1 + prev_proj
        res3 = self.layer3(res2, edge_index)
        res3 = res3 + res2 + prev_proj
        res4 = self.layer4(res3, edge_index)
        res4 = res4 + res3 + prev_proj
        res5 = self.layer5(res4, edge_index)
        res5 = res5 + res4 + prev_proj
        res6 = self.layer6(res5, edge_index)
        res6 = res6 + res5 + prev_proj
        res7 = self.layer7(res6, edge_index)
        res7 = res7 + res6 + prev_proj
        res8 = self.layer8(res7, edge_index)
        res8 = res8 + res7 + prev_proj

        mean = torch.mean(res8, dim=0)

        classes = self.lin_layer(mean)

        return classes

In [56]:
# gat_8layered = BrainSync_8(6,4)
# optimizer = torch.optim.Adam(gat_8layered.parameters(), lr=0.0001)

# X = torch.randn(64, 6)
# edge_index_final = torch.randint(0, 64, (2, 200))
# labels_final = torch.randint(0, 2, (1,)).squeeze() #for the graph not the features

# for epoch in range(100):
#     optimizer.zero_grad()
#     out = gat_8layered(X, edge_index_final)
#     print(out)
#     loss = criterion(out, labels_final)
#     print(labels_final)
#     loss.backward()
#     optimizer.step()

#     if epoch % 10 == 0:
#         print(f"Epoch {epoch}, Loss: {loss.item()}")

In [57]:
# with torch.no_grad():
#     X = torch.randn(64, 6)
#     h = X.clone()
#     for layer in [gat_8layered.layer1, gat_8layered.layer2, gat_8layered.layer3,
#                   gat_8layered.layer4, gat_8layered.layer5, gat_8layered.layer6,
#                   gat_8layered.layer7, gat_8layered.layer8]:
#         h = layer(h, edge_index_final)
#         print(f"Std across nodes: {h.std(dim=0).mean().item():.6f}")

In [58]:
test_layer = GATLayer(6, 4)
X_test = torch.randn(10, 6)
edge_test = torch.randint(0, 10, (2, 20))
out = test_layer(X_test, edge_test)
# print(out) 

In [59]:
gat_8layered_skip = BrainSync_8(6, 4)
optimizer = torch.optim.Adam(gat_8layered_skip.parameters(), lr=0.01)

In [60]:
X = torch.randn(64, 6)

edge_index_final_skip = torch.randint(0, 64, (2, 200))
labels_final_skip = torch.randint(0, 2, (1,)).squeeze() #for the graph not the features

for epoch in range(100):
    optimizer.zero_grad()
    out = gat_8layered_skip(X, edge_index_final_skip)
    loss = criterion(out, labels_final_skip)
    # print(f"out: {out} | loss: {loss}")
    loss.backward()
    optimizer.step()

    if epoch % 10 == 0: 
        print(f"Epoch {epoch}, Loss: {loss.item()}")

Epoch 0, Loss: 3.5965352058410645
Epoch 10, Loss: 0.0007818264421075583
Epoch 20, Loss: 8.106198947643861e-06
Epoch 30, Loss: 1.0728830375228426e-06
Epoch 40, Loss: 4.768370445162873e-07
Epoch 50, Loss: 3.576278118089249e-07
Epoch 60, Loss: 3.576278118089249e-07
Epoch 70, Loss: 2.3841855067985307e-07
Epoch 80, Loss: 2.3841855067985307e-07
Epoch 90, Loss: 2.3841855067985307e-07


In [61]:
with torch.no_grad():
    X = torch.randn(64, 6)
    h = X.clone()
    proj_X = gat_8layered_skip.proj(X)
    
    h = gat_8layered_skip.layer1(h, edge_index_final_skip) + proj_X
    print(f"Layer 1: {h.std(dim=0).mean().item():.6f}")
    
    for i, layer in enumerate([gat_8layered_skip.layer2, gat_8layered_skip.layer3,
                                gat_8layered_skip.layer4, gat_8layered_skip.layer5,
                                gat_8layered_skip.layer6, gat_8layered_skip.layer7,
                                gat_8layered_skip.layer8]):
        prev = h.clone()
        h = layer(h, edge_index_final_skip) + prev
        print(f"Layer {i+2}: {h.std(dim=0).mean().item():.6f}")

Layer 1: 0.641639
Layer 2: 0.679956
Layer 3: 0.763685
Layer 4: 0.815213
Layer 5: 1.028479
Layer 6: 1.075532
Layer 7: 1.061378
Layer 8: 1.150611


## Working with Synthetic Data

### Creating an synthetic dataset similar to DEAP dataset

In [ ]:
eeg_data_arr = np.random.uniform(-100, 100, size=(40,32,8064)) # Gives us the data for 40 trials, using 32 nodes with 8064 timepoints within the 63 sec of data
labels_arr = np.random.randint(1, 10, size=(40,2)) # Gives 40 labels (one for each trial) wth two columns given to arousal and arousal


In [80]:
labels = pd.DataFrame({"Valence": labels_arr[:, 0], "Arousal": labels_arr[:, 1], "Anxiety Quadrant": np.where((labels_arr[:,0] < 5) & (labels_arr[:,1] > 5), 1, 0)})
labels["Anxiety Quadrant"].value_counts()

Anxiety Quadrant
0    33
1     7
Name: count, dtype: int64

### PSD and band power calc

In [64]:
bands = []

for i in range(40):
    for j in range(32):
        freq, psd = scipy.signal.welch(eeg_data_arr[2, 1], fs=128)

        # Masks
        delta_mask = (freq >= 0.5) & (freq <= 4)
        theta_mask = (freq > 4) & (freq <= 8)
        alpha_mask = (freq > 8.0) & (freq <= 13.0)
        beta_mask = (freq > 13) & (freq <= 30)
        gamma_mask = (freq > 30) 

        # Band powers
        delta_band_power = psd[delta_mask].sum()
        theta_band_power = psd[theta_mask].sum()
        alpha_band_power = psd[alpha_mask].sum()
        beta_band_power = psd[beta_mask].sum()
        gamma_band_power = psd[gamma_mask].sum()

        band = [delta_band_power, theta_band_power, alpha_band_power, beta_band_power, gamma_band_power]
        bands.append(band)

eeg_power_data = np.array(bands).reshape(40, 32, 5)

eye_labels = np.random.randint(0, 2, size = (40, 32, 1))

eeg_final = np.concatenate((eeg_power_data, eye_labels), axis=2)
eeg_final.shape


(40, 32, 6)

In [65]:
montage = mne.channels.make_standard_montage('standard_1020')

deap_channels = [
    'AF3', 'AF4', 'Fp1', 'Fp2', 'Fz', 'F3', 'F4', 'F7', 'F8',
    'T7', 'T8', 'FT7', 'FT8', 'TP7', 'TP8',
    'Cz', 'C3', 'C4',
    'Pz', 'P3', 'P4', 'P7', 'P8',
    'Oz', 'O1', 'O2',
    'FCz', 'FC3', 'FC4', 'CPz', 'CP3', 'CP4'
]

positions = np.array([montage.get_positions()['ch_pos'][ch] for ch in deap_channels])

print(len(positions))

def check_neighbors():
    source = []
    neighbors = []

    for electrode in range(32): # Running over each electrode's coordinates
        for neighbor in range(32):
            if neighbor != electrode:
                distance_from_neighbor = 0.0
                for coord in range(len(positions[electrode])): # Running over each coordinate of electrode and the comparison electrode
                    distance_from_neighbor += (positions[electrode][coord] - positions[neighbor][coord]) ** 2
                
                distance_from_neighbor = distance_from_neighbor ** 0.5

            else:
                continue

            # print(f"Distance between: {node_names[electrode]} and {node_names[neighbor]} is: {distance_from_neighbor}")

            if distance_from_neighbor < 0.06:
                # print(f"{deap_channels[electrode]} and {deap_channels[neighbor]} are neighbours")
                source.append(electrode)
                neighbors.append(neighbor)
            
            # else:
            #     print(f"{node_names[electrode]} and {node_names[neighbor]} are not neighbours")
            #     print(distance_from_neighbor)

    return torch.tensor([source, neighbors], dtype=torch.long)

edge_index = check_neighbors()
print(edge_index)
edge_index.shape


32
tensor([[ 0,  0,  0,  0,  1,  1,  2,  2,  2,  3,  3,  3,  4,  4,  4,  4,  5,  5,
          5,  5,  6,  6,  6,  6,  7,  7,  7,  7,  8,  8,  8,  9,  9,  9, 10, 10,
         10, 11, 11, 12, 12, 13, 13, 14, 14, 15, 15, 16, 16, 17, 17, 18, 18, 19,
         19, 20, 21, 21, 21, 22, 22, 22, 23, 23, 24, 24, 24, 25, 25, 25, 26, 26,
         27, 27, 28, 28, 29, 29, 30, 30, 31, 31],
        [ 2,  4,  5,  7,  3,  6,  0,  3,  7,  1,  2,  8,  0,  5,  6, 26,  0,  4,
          7, 27,  1,  4,  8, 28,  0,  2,  5, 11,  3,  6, 12, 11, 13, 21, 12, 14,
         22,  7,  9,  8, 10,  9, 21, 10, 22, 26, 29, 27, 30, 28, 31, 19, 29, 18,
         30, 31,  9, 13, 24, 10, 14, 25, 24, 25, 21, 23, 25, 22, 23, 24,  4, 15,
          5, 16,  6, 17, 15, 18, 16, 19, 17, 20]])


torch.Size([2, 82])

### Creating Data objects for PyG

In [72]:
from torch_geometric.data import Data

dataset = []

for i in range(40): 
    data = Data(x=torch.tensor(eeg_final[i], dtype=torch.float), edge_index=edge_index, y=torch.tensor(labels.iloc[i, 2], dtype=torch.long))
    dataset.append(data)

# dataset

### BrainSync execution

In [75]:
gat_deap = BrainSync_8(6, 4)
optimizer = torch.optim.Adam(gat_deap.parameters(), lr=0.01)

for epoch in range(100):
    for data in dataset:
        optimizer.zero_grad()
        out = gat_deap(data.x, data.edge_index)
        loss = criterion(out, data.y)
        loss.backward()
        optimizer.step()

    if epoch % 10 == 0: 
        print(f"Epoch {epoch}, Loss: {loss.item()}")

Epoch 0, Loss: 0.0
Epoch 10, Loss: 0.2753257751464844
Epoch 20, Loss: 0.0009889479260891676
Epoch 30, Loss: 0.2147912085056305
Epoch 40, Loss: 3.6852617263793945
Epoch 50, Loss: 0.0030906074680387974
Epoch 60, Loss: 0.0008512687054462731
Epoch 70, Loss: 0.030556855723261833
Epoch 80, Loss: 0.15491904318332672
Epoch 90, Loss: 0.201396182179451
